# Karşılaştırmalı Duygu Analizi – BERT Tabanlı Yöntemler

Bu notebook, İngilizce RoBERTa ve Türkçe BERTurk deneylerini tek ve tekrarlanabilir bir akışta yürütür.

## Deney özeti

### İngilizce
- Model: `cardiffnlp/twitter-roberta-base-sentiment-latest`
- Yöntem: Önceden eğitilmiş modelle doğrudan tahmin; bu proje kapsamında fine-tuning yapılmaz.

### Türkçe
- Model: `dbmdz/bert-base-turkish-cased`
- Yöntem: Proje eğitim verisiyle fine-tuning
- Epoch: **2**
- Learning rate: `2e-5`
- Batch size: `16`
- Maksimum dizi uzunluğu: `128`

### Veri ve çıktılar
- İngilizce test: 3.000 örnek (her sınıftan 1.000)
- Türkçe eğitim: 30.000 örnek (her sınıftan 10.000)
- Türkçe test: 3.000 örnek (her sınıftan 1.000)
- Tahmin dosyaları `id, text, true, pred` sütunlarıyla ve UTF-8 kodlamasıyla `results/` klasörüne kaydedilir.

> Not: İngilizce model hazır model olarak kullanılırken Türkçe model proje verisiyle ayrıca eğitilir. Bu nedenle iki dilin sonuçları yorumlanırken bu yöntem farkı dikkate alınmalıdır.

## 1. Ortam kurulumu

Bu hücre gerekli kütüphaneleri kurar. Google Colab'da mümkünse GPU çalışma zamanı seçilmelidir.

In [ ]:
!pip -q install transformers datasets accelerate scikit-learn pandas matplotlib

## 2. Kütüphaneler ve çalışma ortamı

GPU kullanılabiliyorsa Türkçe fine-tuning ve çıkarım işlemleri GPU üzerinde yürütülür. GPU yoksa kod CPU üzerinde de çalışabilir; işlem daha uzun sürebilir.

In [ ]:
import os
import json
import time
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from google.colab import files
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed
)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 0 if torch.cuda.is_available() else -1
print("PyTorch:", torch.__version__)
print("CUDA kullanılabilir mi:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU bulunamadı; CPU kullanılacak.")

## 3. Veri dosyalarının yüklenmesi ve kontrolü

Açılan dosya penceresinden `test_en.csv`, `test_tr.csv` ve `train_tr.csv` dosyalarını birlikte seçin. Her dosyada `id`, `text`, `label` sütunları bulunmalıdır.

In [ ]:
uploaded = files.upload()

required_files = ["test_en.csv", "test_tr.csv", "train_tr.csv"]
missing_files = [name for name in required_files if name not in uploaded]
if missing_files:
    raise FileNotFoundError(f"Eksik dosyalar: {missing_files}")

test_en = pd.read_csv("test_en.csv")
test_tr = pd.read_csv("test_tr.csv")
train_tr = pd.read_csv("train_tr.csv")

required_columns = {"id", "text", "label"}
for name, frame in [
    ("test_en.csv", test_en),
    ("test_tr.csv", test_tr),
    ("train_tr.csv", train_tr)
]:
    if not required_columns.issubset(frame.columns):
        raise ValueError(f"{name} dosyasında gerekli sütunlar bulunamadı: {required_columns}")
    print(f"{name}: {len(frame)} satır")
    print(frame["label"].value_counts())
    print("-" * 45)

os.makedirs("results", exist_ok=True)
os.makedirs("results/figures", exist_ok=True)

## 4. İngilizce RoBERTa ile tahmin

İngilizce test metinleri CardiffNLP RoBERTa duygu modeliyle sınıflandırılır. Modelin İngilizce etiketleri proje sınıf adlarına dönüştürülür ve sonuçlar ortak değerlendirme betiğiyle uyumlu biçimde kaydedilir.

In [ ]:
EN_MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"

english_model = pipeline(
    "sentiment-analysis",
    model=EN_MODEL_NAME,
    device=DEVICE
)

english_texts = test_en["text"].astype(str).tolist()
english_outputs = []

for start in range(0, len(english_texts), 32):
    batch = english_texts[start:start + 32]
    english_outputs.extend(
        english_model(
            batch,
            batch_size=32,
            truncation=True,
            max_length=128
        )
    )

english_label_map = {
    "positive": "Pozitif",
    "negative": "Negatif",
    "neutral": "Nötr"
}

preds_en = pd.DataFrame({
    "id": test_en["id"],
    "text": test_en["text"],
    "true": test_en["label"],
    "pred": [english_label_map[item["label"].lower()] for item in english_outputs]
})

preds_en.to_csv(
    "results/preds_bert_en.csv",
    index=False,
    encoding="utf-8"
)

print("İngilizce tahmin sayısı:", len(preds_en))
print(preds_en["pred"].value_counts())
print("Kaydedildi: results/preds_bert_en.csv")

## 5. Türkçe BERTurk için eğitim ve doğrulama verisinin hazırlanması

Türkçe eğitim verisi, sınıf oranları korunarak (`stratify`) %90 eğitim ve %10 doğrulama olarak ayrılır. Böylece 27.000 örnek eğitim, 3.000 örnek doğrulama için kullanılır. Test kümesi bu aşamada eğitime dahil edilmez.

In [ ]:
TR_MODEL_NAME = "dbmdz/bert-base-turkish-cased"

label2id = {
    "Negatif": 0,
    "Nötr": 1,
    "Pozitif": 2
}
id2label = {value: key for key, value in label2id.items()}

unknown_labels = set(train_tr["label"].dropna().unique()) - set(label2id)
if unknown_labels:
    raise ValueError(f"Eğitim verisinde beklenmeyen etiketler var: {unknown_labels}")

train_tr = train_tr.copy()
train_tr["label_id"] = train_tr["label"].map(label2id)

train_df, val_df = train_test_split(
    train_tr,
    test_size=0.10,
    random_state=SEED,
    stratify=train_tr["label_id"]
)

tokenizer = AutoTokenizer.from_pretrained(TR_MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_dataset = Dataset.from_pandas(
    train_df[["text", "label_id"]].rename(columns={"label_id": "labels"}),
    preserve_index=False
).map(tokenize_batch, batched=True)

val_dataset = Dataset.from_pandas(
    val_df[["text", "label_id"]].rename(columns={"label_id": "labels"}),
    preserve_index=False
).map(tokenize_batch, batched=True)

train_dataset = train_dataset.remove_columns(["text"])
val_dataset = val_dataset.remove_columns(["text"])

print("Eğitim örneği:", len(train_dataset))
print("Doğrulama örneği:", len(val_dataset))

## 6. BERTurk modelinin fine-tuning işlemi

Model iki epoch boyunca eğitilir. Her epoch sonunda doğrulama metrikleri hesaplanır. En iyi doğrulama Macro-F1 değerine sahip checkpoint eğitim sonunda kullanılır.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predicted_ids = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predicted_ids),
        "macro_f1": f1_score(labels, predicted_ids, average="macro")
    }

training_args = TrainingArguments(
    output_dir="/content/berturk_checkpoints",
    num_train_epochs=2,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    seed=SEED,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_macro_f1",
    greater_is_better=True
)

model = AutoModelForSequenceClassification.from_pretrained(
    TR_MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

start_time = time.time()
train_result = trainer.train()
training_time = time.time() - start_time

validation_metrics = trainer.evaluate()

print("Eğitim tamamlandı.")
print("Eğitim süresi (dakika):", round(training_time / 60, 2))
print("Doğrulama metrikleri:", validation_metrics)

## 7. Eğitilmiş modelin kaydedilmesi

En iyi checkpoint'teki model ve tokenizer çalışma alanında saklanır. Model ağırlıkları GitHub deposuna eklenmez; yalnızca yerel/kişisel yedek olarak tutulur.

In [ ]:
FINAL_MODEL_DIR = "/content/berturk_final"
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print("Model ve tokenizer kaydedildi:", FINAL_MODEL_DIR)

## 8. Türkçe test kümesinde tahmin

Eğitim ve doğrulama tamamlandıktan sonra ayrı tutulan 3.000 örneklik test kümesi değerlendirilir. Tahmin dosyası ortak `evaluate.py` ve karşılaştırma notebook'uyla uyumlu olacak şekilde `id, text, true, pred` sütunlarıyla UTF-8 kodlamasında yazılır.

In [ ]:
test_label_ids = test_tr["label"].map(label2id)
if test_label_ids.isna().any():
    raise ValueError("Test verisinde tanımlanmamış sınıf etiketi var.")

test_dataset = Dataset.from_pandas(
    pd.DataFrame({
        "text": test_tr["text"].astype(str),
        "labels": test_label_ids.astype(int)
    }),
    preserve_index=False
).map(tokenize_batch, batched=True)

test_dataset = test_dataset.remove_columns(["text"])

test_output = trainer.predict(test_dataset)
test_pred_ids = np.argmax(test_output.predictions, axis=-1)

preds_tr = pd.DataFrame({
    "id": test_tr["id"],
    "text": test_tr["text"],
    "true": test_tr["label"],
    "pred": [id2label[int(value)] for value in test_pred_ids]
})

preds_tr.to_csv(
    "results/preds_bert_tr.csv",
    index=False,
    encoding="utf-8"
)

print("Türkçe test tahmin sayısı:", len(preds_tr))
print("Tahmin dağılımı:")
print(preds_tr["pred"].value_counts())
print("Kaydedildi: results/preds_bert_tr.csv")

## 9. Türkçe test sonuçlarının metrikleri ve confusion matrix

Accuracy, Macro-F1 ve sınıf bazında precision/recall/F1 değerleri hesaplanır. Confusion matrix görseli `results/figures/` klasörüne kaydedilir.

In [ ]:
y_true = preds_tr["true"]
y_pred = preds_tr["pred"]

test_accuracy = accuracy_score(y_true, y_pred)
test_macro_f1 = f1_score(y_true, y_pred, average="macro")

print(f"Accuracy: {test_accuracy:.4f} ({test_accuracy * 100:.2f}%)")
print(f"Macro-F1: {test_macro_f1:.4f} ({test_macro_f1 * 100:.2f}%)")
print("\nSınıf bazında sonuçlar:")
print(classification_report(
    y_true,
    y_pred,
    labels=["Pozitif", "Negatif", "Nötr"],
    zero_division=0
))

class_order = ["Pozitif", "Negatif", "Nötr"]
cm = confusion_matrix(y_true, y_pred, labels=class_order)

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_order
)
display.plot(cmap="Blues", values_format="d")
plt.title("BERTurk - Türkçe Test Seti")
plt.savefig(
    "results/figures/cm_bert_tr.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

## 10. Eğitim bilgilerinin kaydedilmesi

Eğitim ayarları, süre ve doğrulama/test metrikleri JSON dosyasına kaydedilir. Süre ve metrikler çalıştırılan oturuma göre otomatik doldurulur.

In [ ]:
training_info = {
    "model": TR_MODEL_NAME,
    "epochs": 2,
    "learning_rate": 2e-5,
    "batch_size": 16,
    "max_length": 128,
    "weight_decay": 0.01,
    "seed": SEED,
    "train_samples": len(train_dataset),
    "validation_samples": len(val_dataset),
    "test_samples": len(preds_tr),
    "training_time_minutes": round(training_time / 60, 2),
    "validation_accuracy": round(float(validation_metrics["eval_accuracy"]), 6),
    "validation_macro_f1": round(float(validation_metrics["eval_macro_f1"]), 6),
    "test_accuracy": round(float(test_accuracy), 6),
    "test_macro_f1": round(float(test_macro_f1), 6),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
}

with open("results/berturk_training_info.json", "w", encoding="utf-8") as file:
    json.dump(training_info, file, ensure_ascii=False, indent=4)

print(json.dumps(training_info, ensure_ascii=False, indent=4))

## 11. Son dosya kontrolü

Bu son hücre, beklenen çıktıların oluşturulduğunu ve tahmin CSV'sinin ortak dosya şemasına uyduğunu kontrol eder.

In [ ]:
expected_columns = ["id", "text", "true", "pred"]
saved_tr = pd.read_csv("results/preds_bert_tr.csv", encoding="utf-8")

assert len(saved_tr) == len(test_tr), "Türkçe tahmin satır sayısı test kümesiyle eşleşmiyor."
assert saved_tr.columns.tolist() == expected_columns, "Türkçe tahmin CSV sütunları beklenen formatta değil."
assert saved_tr[["true", "pred"]].notna().all().all(), "Etiket sütunlarında boş değer var."

for output_file in [
    "results/preds_bert_en.csv",
    "results/preds_bert_tr.csv",
    "results/berturk_training_info.json",
    "results/figures/cm_bert_tr.png"
]:
    print(output_file, "->", os.path.exists(output_file))

print("\nTürkçe tahmin CSV kontrolü başarılı.")
print("Accuracy:", round(accuracy_score(saved_tr["true"], saved_tr["pred"]), 4))
print("Macro-F1:", round(f1_score(saved_tr["true"], saved_tr["pred"], average="macro"), 4))